In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
DATA_DIR = os.path.join("..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

In [4]:
df = pd.read_csv('..\data\PS_20174392719_1491204439457_log.csv')

In [5]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [6]:
df.shape

(6362620, 11)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [8]:
df.isna().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [9]:
df.duplicated().sum()

0

In [10]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [11]:
print(df['nameOrig'].nunique())
print(df['nameDest'].nunique())
print("Fraud rate:", df['isFraud'].mean())
print("Fraud percentage:", df['isFraud'].mean() * 100)

6353307
2722362
Fraud rate: 0.001290820448180152
Fraud percentage: 0.12908204481801522


In [12]:
df['isFraud'].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [13]:
orig_repeat = df.groupby('nameOrig').size()
print(orig_repeat.value_counts())
print("Maksimum təkrar sayı:", orig_repeat.max())

1    6344009
2       9283
3         15
Name: count, dtype: int64
Maksimum təkrar sayı: 3


In [14]:
df['balance_mismatch'] = ((df['oldbalanceOrg'] - df['amount']).round(2)!= df['newbalanceOrig'].round(2)).astype(int)
print(df['balance_mismatch'].value_counts())

balance_mismatch
1    5125552
0    1237068
Name: count, dtype: int64


In [15]:
fraud_customers = df.loc[df['isFraud'] == 1, 'nameOrig'].unique()
clean_customers = (df.loc[~df['nameOrig'].isin(fraud_customers), 'nameOrig'].drop_duplicates().sample(3000, random_state=42))
selected_customers = set(fraud_customers) | set(clean_customers)

subset = df[df['nameOrig'].isin(selected_customers)].copy()

In [16]:
print("Original shape:", df.shape)
print("Subset shape:", subset.shape)

print("Original fraud rate:", df['isFraud'].mean())
print("Subset fraud rate:", subset['isFraud'].mean())

Original shape: (6362620, 12)
Subset shape: (11247, 12)
Original fraud rate: 0.001290820448180152
Subset fraud rate: 0.730239174891082


## 5. Entity və event timestamp

In [17]:
base_date = pd.Timestamp("2024-01-01")
subset['event_timestamp'] = (base_date + pd.to_timedelta(subset['step'], unit='h'))
subset = subset.sort_values(['nameOrig', 'event_timestamp']).reset_index(drop=True)

subset[['nameOrig', 'step', 'event_timestamp']].head()

,nameOrig,step,event_timestamp
0,C1000036340,655,2024-01-28 07:00:00
1,C1000086512,95,2024-01-04 23:00:00
2,C1000331499,262,2024-01-11 22:00:00
3,C1000437286,284,2024-01-12 20:00:00
4,C1000484178,504,2024-01-22 00:00:00


In [18]:
step_agg = (subset.groupby(['nameOrig', 'step']).agg(step_txn_count=('amount', 'size'),step_amount_sum=('amount', 'sum'),step_last_amount=('amount', 'last'),).reset_index().sort_values(['nameOrig', 'step']))

step_agg.head()

,nameOrig,step,step_txn_count,step_amount_sum,step_last_amount
0,C1000036340,655,1,253648.68,253648.68
1,C1000086512,95,1,33676.59,33676.59
2,C1000331499,262,1,2016790.84,2016790.84
3,C1000437286,284,1,181182.79,181182.79
4,C1000484178,504,1,3018810.85,3018810.85


In [19]:
g_step = step_agg.groupby('nameOrig')

step_agg['cum_count_incl_current']  = g_step['step_txn_count'].cumsum()
step_agg['cum_amount_incl_current'] = g_step['step_amount_sum'].cumsum()

step_agg['txn_count_so_far']  = step_agg['cum_count_incl_current']  - step_agg['step_txn_count']
step_agg['total_sent_so_far'] = step_agg['cum_amount_incl_current'] - step_agg['step_amount_sum']

step_agg['avg_amount_so_far'] = np.where(step_agg['txn_count_so_far'] > 0,step_agg['total_sent_so_far'] / step_agg['txn_count_so_far'],0.0)

In [20]:
step_agg['last_txn_amount'] = g_step['step_last_amount'].shift(1).fillna(0)
step_agg['prev_step'] = g_step['step'].shift(1)

step_agg['hours_since_last_txn'] = (step_agg['step'] - step_agg['prev_step']).fillna(0)

In [21]:
feature_col_step = ['nameOrig', 'step', 'txn_count_so_far', 'total_sent_so_far', 'avg_amount_so_far', 'last_txn_amount', 'hours_since_last_txn']
subset = subset.merge(step_agg[feature_col_step], on=['nameOrig', 'step'], how='left')
subset.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_mismatch,event_timestamp,txn_count_so_far,total_sent_so_far,avg_amount_so_far,last_txn_amount,hours_since_last_txn
0,655,TRANSFER,253648.68,C1000036340,253648.68,0.0,C1958275811,0.00,0.00,1,0,0,2024-01-28 07:00:00,0,0.0,0.0,0.0,0.0
1,95,CASH_OUT,33676.59,C1000086512,33676.59,0.0,C1759363094,0.00,33676.59,1,0,0,2024-01-04 23:00:00,0,0.0,0.0,0.0,0.0
2,262,TRANSFER,2016790.84,C1000331499,2016790.84,0.0,C1778895918,0.00,0.00,1,0,0,2024-01-11 22:00:00,0,0.0,0.0,0.0,0.0
3,284,CASH_OUT,181182.79,C1000437286,9601.00,0.0,C1827183939,150236.44,331419.24,0,0,1,2024-01-12 20:00:00,0,0.0,0.0,0.0,0.0
4,504,CASH_OUT,3018810.85,C1000484178,3018810.85,0.0,C895750711,65126.98,3083937.83,1,0,0,2024-01-22 00:00:00,0,0.0,0.0,0.0,0.0


### Leakage

In [22]:
first_step = step_agg.groupby('nameOrig').head(1)

assert (first_step['txn_count_so_far'] == 0).all()
assert (first_step['avg_amount_so_far'] == 0).all()
assert (first_step['total_sent_so_far'] == 0).all()
assert (first_step['last_txn_amount'] == 0).all()
assert (first_step['hours_since_last_txn'] == 0).all()

print('Test 1 passed')

Test 1 passed


In [23]:
same_step = subset.groupby(['nameOrig','step'])[['txn_count_so_far', 'total_sent_so_far', 'avg_amount_so_far', 'last_txn_amount', 'hours_since_last_txn']].nunique()
assert (same_step <=1).all().all()

print('Test 2 passed')

Test 2 passed


In [24]:
feature_cols = ['nameOrig','event_timestamp','txn_count_so_far', 'total_sent_so_far', 'avg_amount_so_far', 'last_txn_amount', 'hours_since_last_txn','balance_mismatch']

feature_df = subset[feature_cols].copy()
feature_df.to_parquet(os.path.join(DATA_DIR,'transaction_features.parquet'),index = False)
feature_df.head()

,nameOrig,event_timestamp,txn_count_so_far,total_sent_so_far,avg_amount_so_far,last_txn_amount,hours_since_last_txn,balance_mismatch
0,C1000036340,2024-01-28 07:00:00,0,0.0,0.0,0.0,0.0,0
1,C1000086512,2024-01-04 23:00:00,0,0.0,0.0,0.0,0.0,0
2,C1000331499,2024-01-11 22:00:00,0,0.0,0.0,0.0,0.0,0
3,C1000437286,2024-01-12 20:00:00,0,0.0,0.0,0.0,0.0,1
4,C1000484178,2024-01-22 00:00:00,0,0.0,0.0,0.0,0.0,0


In [25]:
entity_df = subset[['nameOrig', 'event_timestamp', 'isFraud']].copy()
entity_df.to_parquet(os.path.join(DATA_DIR,'entity_df.parquet'),index = False)
entity_df.head()

,nameOrig,event_timestamp,isFraud
0,C1000036340,2024-01-28 07:00:00,1
1,C1000086512,2024-01-04 23:00:00,1
2,C1000331499,2024-01-11 22:00:00,1
3,C1000437286,2024-01-12 20:00:00,0
4,C1000484178,2024-01-22 00:00:00,1


## Feature Documentation

This section documents the meaning, calculation logic, and rationale for every feature produced by this pipeline.

### Entity and timestamp

- **`nameOrig`** — the entity key. Represents the sender's customer ID. All historical features below are computed per `nameOrig`.
- **`event_timestamp`** — the point-in-time reference used for feature retrieval. PaySim's `step` field represents simulated hours rather than a real-world timestamp, so it is converted into an absolute datetime (`base_date + step hours`) to satisfy Feast's requirement for a valid timestamp column. The reference date itself is arbitrary and does not represent the actual transaction date — only the relative time ordering matters.

### Historical (point-in-time safe) features

All features below are computed using **strictly prior `step` values only**. Because multiple transactions from the same sender can share the same `step` (i.e. occur within the same simulated hour), features are aggregated at the `(nameOrig, step)` level first, then merged back onto individual transactions. This guarantees that transactions sharing a step never leak information into each other's features — each transaction only "sees" information from steps that occurred strictly before it.

- **`txn_count_so_far`** — the number of transactions the sender made in all previous steps (not including the current step). Indicates how established or active the account is; new or low-activity accounts can carry higher fraud risk.
- **`avg_amount_so_far`** — the average transaction amount across all of the sender's previous steps. Used to detect whether the current transaction deviates sharply from the sender's typical behavior.
- **`total_sent_so_far`** — the cumulative sum of transaction amounts across all of the sender's previous steps, regardless of transaction type. Reflects the sender's overall prior transaction volume.
- **`last_txn_amount`** — the transaction amount from the sender's most recent *prior* step. Used to detect sudden spikes relative to the last known activity. Note: if the previous step contained multiple transactions, the last one (by row order within that step) is used as an approximation, since the true intra-step order is unknown.
- **`hours_since_last_txn`** — the number of hours (steps) elapsed since the sender's previous distinct step. Used to detect unusually frequent transaction bursts, which can be a fraud indicator.

### Current-event feature

- **`balance_mismatch`** — a binary flag (0/1) indicating whether the sender's balance after the transaction (`newbalanceOrig`) is inconsistent with the expected balance (`oldbalanceOrg - amount`). Unlike the features above, this is **not a historical feature** — it is derived entirely from the current transaction's own data. It does not use future information, but it also does not represent "the past"; it is a data-quality / current-event signal computed at prediction time.

### Label

- **`isFraud`** — the prediction target. Included only in `entity_df.parquet`, not in the feature Parquet file, since it is a label rather than a feature.

### Leakage prevention summary

Historical features are computed by aggregating transaction counts and amounts at the step level, taking a cumulative sum **including** the current step, then subtracting the current step's own contribution. What remains reflects only strictly prior steps. This approach was validated with three checks:
1. Every sender's first step has all historical features equal to zero.
2. Transactions sharing the same `(nameOrig, step)` have identical historical feature values (proving no leakage occurs between transactions within the same step).
3. A manually computed feature set (using only `step < current_step`) matches the pipeline's output for a sample customer.

### Sampling note

The dataset was reduced to a customer-level subset containing all fraud-associated customers plus 3,000 randomly sampled non-fraud customers, preserving each selected customer's full transaction history (required for historical feature computation). This is a label-aware sampling strategy: fraud rate in the subset is therefore not representative of the original ~0.129% population fraud rate, and the subset is intended for feature-pipeline development and validation, not for estimating real-world fraud prevalence.